# Train, Validation, and Test Splits

## Overview

A machine learning model should not only perform well on the data it was trained on; it should also generalize to new, unseen data.

In this notebook, the evaluation workflow is separated into three distinct roles:

- **Training set:** used to fit model parameters.
- **Validation set:** used during development to compare settings and model choices.
- **Test set:** reserved for the final evaluation after development decisions are complete.

The notebook also keeps a short record of the development process. Some attempts improve the validation result and others do not. The important rule is that these decisions are made using the validation set rather than repeatedly checking the final test set.

## Objectives

By the end of this notebook, we will:

1. Create a stratified 60/20/20 train, validation, and test split.
2. Show why accuracy alone is misleading for this imbalanced target.
3. Train an initial Decision Tree and improve one of its settings using validation data.
4. Compare a small set of model families from previous supervised-learning work.
5. Make one additional validation-based decision to improve the selected model.
6. Freeze the final configuration before opening the test set.
7. Evaluate the final model once on the held-out test set.
8. Explain why repeated tuning against the test set would make the reported result misleading.


## 1. Importing the Required Libraries

The notebook uses a small set of tools for data handling, splitting, modeling, and evaluation.

- **Pandas** is used to load and inspect the dataset.
- **`train_test_split`** creates the train, validation, and test subsets.
- **Decision Tree, Logistic Regression, and Random Forest** provide different supervised-learning approaches for the development comparison.
- **Accuracy, F1-score, precision, recall, and a confusion matrix** are used to evaluate and interpret classification performance.

The test set is not used for model-selection decisions.


In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)


## 2. Load the COVID-19 Classification Dataset

This notebook reuses the COVID-19 testing dataset from Week 3 so that the focus remains on evaluation methodology rather than introducing a new modeling problem.

The dataset is stored in the Week 3 dataset directory and is loaded as a CSV file using Pandas. At this stage, the data is loaded without applying model-related preprocessing.


In [2]:
DATA_PATH = "../../week-3/corona dataset/corona_tested_individuals_ver_006.english.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)

df.head()


,test_date,cough,fever,sore_throat,shortness_of_breath,head_ache,corona_result,age_60_and_above,gender,test_indication
0,2020-04-30,0.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
1,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
2,2020-04-30,0.0,1.0,0.0,0.0,0.0,negative,NaN,male,Other
3,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,female,Other
4,2020-04-30,1.0,0.0,0.0,0.0,0.0,negative,NaN,male,Other


## 3. Verify the Dataset Before Splitting

We perform only a focused inspection here rather than repeating the full EDA from previous weeks.

The checks are limited to information that affects the evaluation setup:

- dataset size and columns,
- target distribution,
- missing values.

The target distribution is especially important because the positive class is much smaller than the negative class.


In [3]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nTarget counts:")
print(df["corona_result"].value_counts(dropna=False))

print("\nTarget percentages:")
print(
    (df["corona_result"].value_counts(normalize=True, dropna=False) * 100)
    .round(2)
)

print("\nMissing values:")
print(df.isna().sum())


Dataset shape: (278848, 10)

Columns:
['test_date', 'cough', 'fever', 'sore_throat', 'shortness_of_breath', 'head_ache', 'corona_result', 'age_60_and_above', 'gender', 'test_indication']

Target counts:
corona_result
negative    260227
positive     14729
other         3892
Name: count, dtype: int64

Target percentages:
corona_result
negative    93.32
positive     5.28
other        1.40
Name: proportion, dtype: float64

Missing values:
test_date                   0
cough                     252
fever                     252
sore_throat                 1
shortness_of_breath         1
head_ache                   1
corona_result               0
age_60_and_above       127320
gender                  19563
test_indication             0
dtype: int64


## 4. Prepare the Binary Classification Data

The raw target contains `negative`, `positive`, and `other`. For this notebook, the task is defined as binary classification between confirmed negative and positive results.

The `other` target category is excluded, and the remaining classes are encoded as:

- `negative` → 0
- `positive` → 1

The feature set contains the five recorded symptom indicators together with `test_indication`. The test indication is assumed to be available before the COVID-19 result is known.

Because `test_indication` is categorical, it is represented with two binary indicators:

- contact with a confirmed case,
- travel abroad.

The remaining `Other` category is represented when both indicators are zero.

Rows missing one of the selected symptom values are removed. Only a very small number of rows are affected, so this keeps the Day 1 preprocessing simple without introducing an imputation strategy.


In [4]:
symptom_features = [
    "cough",
    "fever",
    "sore_throat",
    "shortness_of_breath",
    "head_ache",
]

target_column = "corona_result"

model_df = df[
    df[target_column].isin(["negative", "positive"])
].copy()

rows_before_missing_removal = len(model_df)

model_df = model_df.dropna(subset=symptom_features)

# Fixed categorical representation; no statistics are learned from the data.
model_df["contact_with_confirmed"] = (
    model_df["test_indication"] == "Contact with confirmed"
).astype(int)

model_df["abroad"] = (
    model_df["test_indication"] == "Abroad"
).astype(int)

feature_columns = symptom_features + [
    "contact_with_confirmed",
    "abroad",
]

X = model_df[feature_columns].astype(int)

y = (
    model_df[target_column]
    .map({"negative": 0, "positive": 1})
    .astype(int)
)

print("Rows before removing missing symptom values:", rows_before_missing_removal)
print("Rows available for modeling:", len(model_df))
print("Rows removed:", rows_before_missing_removal - len(model_df))

print("\nFeatures:")
print(feature_columns)

print("\nFeature matrix shape:", X.shape)
print("Target shape:", y.shape)

print("\nEncoded target distribution:")
print(y.value_counts())


Rows before removing missing symptom values: 274956
Rows available for modeling: 274702
Rows removed: 254

Features:
['cough', 'fever', 'sore_throat', 'shortness_of_breath', 'head_ache', 'contact_with_confirmed', 'abroad']

Feature matrix shape: (274702, 7)
Target shape: (274702,)

Encoded target distribution:
corona_result
0    260008
1     14694
Name: count, dtype: int64


## 5. Create the Train, Validation, and Test Sets

The modeling data is divided into three subsets:

- **60% Training:** used to fit model parameters.
- **20% Validation:** used to compare settings and model choices.
- **20% Test:** reserved for the final evaluation.

The split is created in two stages. First, 20% is held out as test data. The remaining 80% is then split so that 25% of that temporary set becomes validation data. This produces the final 60/20/20 proportions.

Stratification is used in both calls so that the class proportions remain approximately consistent during development.


In [5]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=42,
    stratify=y_temp,
)


## 6. Verify the Split

A successful function call does not automatically prove that the experimental setup is correct.

We verify:

1. that the observations were allocated according to the intended 60/20/20 proportions, and
2. that the training and validation sets preserve the positive-class rate of the full modeling data.

The test-set size is checked at this stage, but its labels are not inspected during model selection.


In [6]:
total_samples = len(X)

split_sizes = pd.DataFrame({
    "Samples": [
        len(X_train),
        len(X_val),
        len(X_test),
    ],
    "Dataset Percentage": [
        len(X_train) / total_samples * 100,
        len(X_val) / total_samples * 100,
        len(X_test) / total_samples * 100,
    ],
}, index=["Train", "Validation", "Test"])

development_balance = pd.DataFrame({
    "Positive Rate": [
        y.mean() * 100,
        y_train.mean() * 100,
        y_val.mean() * 100,
    ],
}, index=["Full Modeling Data", "Train", "Validation"])

print("Split sizes:")
print(split_sizes.round(2))

print("\nPositive-class rate:")
print(development_balance.round(2))


Split sizes:
            Samples  Dataset Percentage
Train        164820                60.0
Validation    54941                20.0
Test          54941                20.0

Positive-class rate:
                    Positive Rate
Full Modeling Data           5.35
Train                        5.35
Validation                   5.35


### Split Verification

The split follows the intended 60/20/20 allocation.

The training and validation sets also preserve the positive-class rate of the full modeling dataset, which confirms that stratification worked as expected for the development data.

From this point until the final evaluation, model decisions will be based on the training and validation sets only.


## 7. Define the Validation Metric

The target is strongly imbalanced, so accuracy alone can give a misleading impression of model quality.

For this experiment:

- **F1-score** is the primary metric for model-selection decisions.
- **Accuracy** is reported as secondary context.
- **Precision and recall** are used later to understand the type of errors being made.

The selection metric is defined before comparing models so that the rule is not changed after seeing the results.


### Majority-Class Sanity Check

Before training a real model, we establish a simple reference point.

A classifier that predicts every observation as negative can achieve high accuracy because the negative class dominates the dataset. Its F1-score, however, should reveal that it identifies no positive cases.


In [7]:
majority_predictions = [0] * len(y_val)

majority_accuracy = accuracy_score(
    y_val,
    majority_predictions,
)

majority_f1 = f1_score(
    y_val,
    majority_predictions,
    zero_division=0,
)

print(f"Majority-class Accuracy: {majority_accuracy:.4f}")
print(f"Majority-class F1-score: {majority_f1:.4f}")


Majority-class Accuracy: 0.9465
Majority-class F1-score: 0.0000


### Majority Baseline Interpretation

The majority-class baseline achieves high accuracy while its F1-score is zero. It correctly predicts many negative cases simply because negatives are common, but it never identifies the positive class.

This confirms that accuracy alone is not a suitable model-selection metric for this experiment.


## 8. Start With a Small Decision Tree

The first real model is a Decision Tree with `max_depth=2`.

This gives us a simple starting point. The model is fitted on the training set and evaluated on the validation set. If the validation result is not satisfactory, the next development decision can be made without consulting the final test set.


In [8]:
baseline_model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42,
)

baseline_model.fit(X_train, y_train)

baseline_val_predictions = baseline_model.predict(X_val)

baseline_accuracy = accuracy_score(
    y_val,
    baseline_val_predictions,
)

baseline_f1 = f1_score(
    y_val,
    baseline_val_predictions,
)

print(f"Validation Accuracy: {baseline_accuracy:.4f}")
print(f"Validation F1-score: {baseline_f1:.4f}")


Validation Accuracy: 0.9653
Validation F1-score: 0.6363


### Initial Decision Tree Result

The small tree is clearly better than the majority baseline on F1-score, but there is still room to improve.

Instead of changing several things at once, the next experiment changes only the tree depth. This keeps the comparison easy to interpret.


## 9. Tune the Decision Tree Depth on Validation Data

Several values of `max_depth` are compared.

Every candidate:

1. is trained on the same training set,
2. is evaluated on the same validation set,
3. is compared using validation F1-score as the primary criterion.

The test set is excluded from this process.


In [9]:
candidate_depths = [2, 3, 4, 5, 6, 8, None]

validation_results = []

for depth in candidate_depths:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42,
    )

    model.fit(X_train, y_train)
    val_predictions = model.predict(X_val)

    validation_results.append({
        "max_depth": str(depth),
        "validation_accuracy": accuracy_score(y_val, val_predictions),
        "validation_f1": f1_score(y_val, val_predictions),
    })

validation_results_df = pd.DataFrame(validation_results)

validation_results_df.round(4)


,max_depth,validation_accuracy,validation_f1
0,2,0.9653,0.6363
1,3,0.9663,0.6558
2,4,0.9671,0.6345
3,5,0.9675,0.6320
4,6,0.9678,0.6402
5,8,0.9677,0.6388
6,None,0.9677,0.6388


### Decision Tree Tuning Result

Increasing the depth from 2 to 3 improves validation F1-score. Deeper trees achieve slightly higher accuracy in some cases, but their F1-scores are lower.

Because F1-score was chosen in advance as the primary metric, `max_depth=3` is retained as the best Decision Tree configuration.

At this point, the tree has improved, but the result is still not high enough to assume that tree depth is the only limitation. The next step is to test whether another model family handles the same features better.


## 10. Compare a Few Model Families

The tuned Decision Tree is now compared with two other supervised-learning models:

- **Logistic Regression**
- **Random Forest**

This is a small development comparison rather than exhaustive tuning. All models use the same training data, the same validation data, and the same features.

A weaker attempt is still useful: if a more complex model fails to improve the chosen metric, that result helps rule out the assumption that model complexity alone is the answer.


In [10]:
candidate_models = {
    "Tuned Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        random_state=42,
    ),
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
    ),
}

model_comparison_results = []

for model_name, model in candidate_models.items():
    model.fit(X_train, y_train)
    val_predictions = model.predict(X_val)

    model_comparison_results.append({
        "model": model_name,
        "validation_accuracy": accuracy_score(y_val, val_predictions),
        "validation_precision": precision_score(y_val, val_predictions),
        "validation_recall": recall_score(y_val, val_predictions),
        "validation_f1": f1_score(y_val, val_predictions),
    })

model_comparison_df = (
    pd.DataFrame(model_comparison_results)
    .sort_values("validation_f1", ascending=False)
    .reset_index(drop=True)
)

model_comparison_df.round(4)


,model,validation_accuracy,validation_precision,validation_recall,validation_f1
0,Logistic Regression,0.9665,0.7270,0.5988,0.6567
1,Tuned Decision Tree,0.9663,0.7231,0.5999,0.6558
2,Random Forest,0.9678,0.7962,0.5356,0.6404


### Model Comparison Interpretation

Logistic Regression produces the highest validation F1-score, but only by a small margin over the tuned Decision Tree.

Random Forest reaches the highest validation accuracy and precision, yet its recall is lower. As a result, its F1-score is worse than both Logistic Regression and the tuned tree.

This is a useful failed improvement attempt: a more complex model did not automatically produce a better result for the metric we care about.

Logistic Regression is therefore the leading candidate, but the margin is small. Before opening the test set, we make one more validation-only decision: whether its default classification threshold gives the best precision-recall balance for this experiment.


## 11. Adjust the Logistic Regression Classification Threshold

Logistic Regression outputs a probability for the positive class. By default, a probability of 0.50 or greater is classified as positive.

Because the positive class is uncommon, the default threshold is not guaranteed to give the best F1-score. We compare a small set of thresholds using validation data only.

Lowering the threshold usually increases recall because the model predicts positive more often, but it can also reduce precision. The goal is to find a better balance rather than simply maximizing one metric.


In [11]:
logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

logistic_model.fit(X_train, y_train)

val_positive_probability = logistic_model.predict_proba(X_val)[:, 1]

candidate_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]

threshold_results = []

for threshold in candidate_thresholds:
    threshold_predictions = (
        val_positive_probability >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "validation_accuracy": accuracy_score(y_val, threshold_predictions),
        "validation_precision": precision_score(y_val, threshold_predictions),
        "validation_recall": recall_score(y_val, threshold_predictions),
        "validation_f1": f1_score(y_val, threshold_predictions),
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df.round(4)


,threshold,validation_accuracy,validation_precision,validation_recall,validation_f1
0,0.1,0.9413,0.4652,0.6529,0.5433
1,0.2,0.9666,0.7157,0.6237,0.6665
2,0.3,0.9664,0.7213,0.6067,0.6590
3,0.4,0.9665,0.7270,0.5988,0.6567
4,0.5,0.9665,0.7270,0.5988,0.6567


### Threshold Tuning Interpretation

The lowest threshold is too aggressive: recall increases, but precision falls enough to reduce F1-score.

A threshold of `0.20` gives the best validation F1-score among the tested values. It improves recall while preserving enough precision to produce a better overall balance than the default threshold.

This threshold is selected using validation data only. The model family and threshold are now fixed before the test set is opened.


## 12. Development Summary Before Final Testing

The development path now contains both successful and unsuccessful attempts:

- the majority baseline showed why accuracy is misleading,
- a small Decision Tree provided a real starting point,
- tuning `max_depth` improved the tree,
- Logistic Regression slightly improved validation F1-score,
- Random Forest did not improve the chosen metric despite higher accuracy,
- validation-based threshold adjustment produced the best development F1-score.

This is the point where development stops. Continuing to make decisions after looking at the test result would change the role of the test set.


In [12]:
selected_tree_f1 = validation_results_df.loc[
    validation_results_df["max_depth"] == "3",
    "validation_f1",
].iloc[0]

logistic_default_f1 = model_comparison_df.loc[
    model_comparison_df["model"] == "Logistic Regression",
    "validation_f1",
].iloc[0]

random_forest_f1 = model_comparison_df.loc[
    model_comparison_df["model"] == "Random Forest",
    "validation_f1",
].iloc[0]

selected_threshold = 0.20

selected_threshold_f1 = threshold_results_df.loc[
    threshold_results_df["threshold"] == selected_threshold,
    "validation_f1",
].iloc[0]

development_summary = pd.DataFrame({
    "Stage": [
        "Majority-class baseline",
        "Initial Decision Tree (depth=2)",
        "Tuned Decision Tree (depth=3)",
        "Logistic Regression (threshold=0.50)",
        "Random Forest",
        "Logistic Regression (threshold=0.20)",
    ],
    "Validation F1": [
        majority_f1,
        baseline_f1,
        selected_tree_f1,
        logistic_default_f1,
        random_forest_f1,
        selected_threshold_f1,
    ],
})

development_summary.round(4)


,Stage,Validation F1
0,Majority-class baseline,0.0000
1,Initial Decision Tree (depth=2),0.6363
2,Tuned Decision Tree (depth=3),0.6558
3,Logistic Regression (threshold=0.50),0.6567
4,Random Forest,0.6404
5,Logistic Regression (threshold=0.20),0.6665


## 13. Freeze the Final Model Before Test Evaluation

The final development decision is:

- **Model:** Logistic Regression
- **Classification threshold:** 0.20

The Logistic Regression model is fitted on the training set only. The validation set was used for development decisions, while the test set has not been used to choose the model family or threshold.

From this point forward, the configuration is fixed.


In [13]:
selected_model_name = "Logistic Regression"
selected_threshold = 0.20

final_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
)

final_model.fit(X_train, y_train)

print("Selected model:", selected_model_name)
print(f"Selected classification threshold: {selected_threshold:.2f}")


Selected model: Logistic Regression
Selected classification threshold: 0.20


## 14. Evaluate the Final Model on the Test Set

All model-selection decisions are complete.

The held-out test set is now used once to estimate how the finalized development choice performs on data that did not influence model fitting or model selection.

The test result is reported rather than used as another opportunity for tuning.


In [14]:
test_positive_probability = final_model.predict_proba(X_test)[:, 1]

test_predictions = (
    test_positive_probability >= selected_threshold
).astype(int)

test_accuracy = accuracy_score(
    y_test,
    test_predictions,
)

test_f1 = f1_score(
    y_test,
    test_predictions,
)

print("Final model:", selected_model_name)
print(f"Classification threshold: {selected_threshold:.2f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test F1-score: {test_f1:.4f}")


Final model: Logistic Regression
Classification threshold: 0.20
Test Accuracy: 0.9664
Test F1-score: 0.6656


## 15. Examine the Final Error Profile

Accuracy and F1-score summarize performance, but they do not show which type of mistake is more common.

Precision, recall, and the confusion matrix are used to interpret the final result. This analysis describes the finalized model; it is not used to reopen model selection.


In [15]:
test_precision = precision_score(y_test, test_predictions)
test_recall = recall_score(y_test, test_predictions)
test_confusion_matrix = confusion_matrix(y_test, test_predictions)

print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

print("\nConfusion Matrix:")
print(test_confusion_matrix)


Test Precision: 0.7122
Test Recall: 0.6247

Confusion Matrix:
[[51260   742]
 [ 1103  1836]]


### Final Error Analysis

The final model identifies a meaningful portion of positive cases, but false negatives remain important.

The selected threshold improves recall compared with the default Logistic Regression decision while maintaining enough precision to improve F1-score. Even so, the remaining missed positive cases are a clear limitation.

These results demonstrate a sound evaluation workflow; they should not be interpreted as evidence that this classifier is ready for real diagnostic use.


## 16. Compare Validation and Test Performance

The selected model and threshold were chosen using validation data. The test set was kept outside that decision process.

We now compare validation and test metrics to see whether the development result was reasonably representative of the final held-out evaluation.


In [16]:
final_val_predictions = (
    final_model.predict_proba(X_val)[:, 1] >= selected_threshold
).astype(int)

final_val_accuracy = accuracy_score(
    y_val,
    final_val_predictions,
)

final_val_f1 = f1_score(
    y_val,
    final_val_predictions,
)

final_evaluation = pd.DataFrame({
    "Accuracy": [
        final_val_accuracy,
        test_accuracy,
    ],
    "F1-score": [
        final_val_f1,
        test_f1,
    ],
}, index=["Validation", "Test"])

final_evaluation.round(4)


,Accuracy,F1-score
Validation,0.9666,0.6665
Test,0.9664,0.6656


### Validation vs. Test Interpretation

The validation and test F1-scores are very close, which suggests that the development result was reasonably representative for this particular split.

That does not prove the model will perform identically on all future data. It also does not prove that the small difference between candidate model families is stable.

A single validation set can still be lucky or unlucky, which is why cross-validation is the natural next step.


## 17. Why the Test Set Must Remain Untouched

The test set is useful only while it stays outside the development feedback loop.

During this notebook, the validation set influenced decisions such as tree depth, model family, and classification threshold. That is acceptable because validation data exists for this purpose.

If the same decisions were repeatedly changed after looking at test performance, the engineer would begin adapting the system to that particular test set. The model might never be fitted directly on test observations, but test information would still influence the final system through the engineer's decisions.

At that point, the test set would effectively become another validation set and its score would no longer be an independent final estimate.


## 18. Conclusion

This notebook demonstrated a complete train/validation/test workflow for an imbalanced COVID-19 classification problem.

The data was divided into stratified 60% training, 20% validation, and 20% test subsets. A majority-class baseline first showed why high accuracy alone is misleading for this target, so F1-score was defined in advance as the primary development metric.

Development then progressed in small steps. A Decision Tree was trained, its depth was tuned on validation data, and several model families were compared. Logistic Regression slightly outperformed the tuned tree on validation F1-score, while Random Forest achieved higher accuracy but a lower F1-score. This showed that a more complex model does not automatically improve the metric that matters.

A final validation-only threshold experiment improved the Logistic Regression F1-score further. Only after the model family and threshold were fixed was the held-out test set used for final evaluation.

The final validation and test results are close, but the remaining false negatives still limit the model. More reliable model comparison requires repeated validation rather than trusting one split, which motivates cross-validation in the next stage.

The main lesson is not simply which model scored highest. It is that development decisions belong on training and validation data, while the test set should remain outside that feedback loop until the end.
